# 🦷 Session 2 — Simple 3D CBCT Lab

## Real course CBCT data

This short lab uses a **separate workshop copy** of the CBCT data stored in the Session 2 course resources.

Course dataset folder:
https://drive.google.com/drive/folders/1uZuAxSDNOgDEVt3NFVWhfw9ISwlQudup

It contains 9 de-identified teaching cases:

**Case_001 → Case_009**

Each case contains only:
- `image.nii.gz` — 3D CBCT volume
- `label.nii.gz` — matching 3D segmentation label

### What we will do

**Mount Drive → choose one case → load CBCT → view 3 planes → overlay the segmentation → show patient-level splitting**

> Educational workshop use only. This notebook does not access the original research-project folder.

## 1 — Mount Google Drive

Run this cell and approve access.

If the course folder is shared with you but is not visible under **My Drive**, open the course dataset link above and choose **Organize → Add shortcut to Drive**, then rerun the notebook.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2 — Install/import the only packages we need

In [ ]:
!pip -q install nibabel

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib

## 3 — Find the course CBCT folder

The notebook searches your mounted **My Drive** for:

`CBCT_3D_Workshop_Data`

No research-project path is used.

In [ ]:
COURSE_FOLDER_NAME = "CBCT_3D_Workshop_Data"
COURSE_FOLDER_URL = "https://drive.google.com/drive/folders/1uZuAxSDNOgDEVt3NFVWhfw9ISwlQudup"

matches = list(Path("/content/drive/MyDrive").rglob(COURSE_FOLDER_NAME))

if not matches:
    raise FileNotFoundError(
        f"{COURSE_FOLDER_NAME} was not found in My Drive.\n"
        f"Open the course folder: {COURSE_FOLDER_URL}\n"
        "Then choose Organize → Add shortcut to Drive and rerun this cell."
    )

DATA_DIR = matches[0]

cases = sorted([
    p for p in DATA_DIR.glob("Case_*")
    if (p / "image.nii.gz").exists() and (p / "label.nii.gz").exists()
])

print("Course folder:", DATA_DIR)
print("Complete cases found:", len(cases))
print([p.name for p in cases])

## 4 — Choose one case

Start with **Case_001**.  
You can change the dropdown later.

In [ ]:
CASE = "Case_001"  # @param ["Case_001","Case_002","Case_003","Case_004","Case_005","Case_006","Case_007","Case_008","Case_009"]

case_dir = DATA_DIR / CASE
image_path = case_dir / "image.nii.gz"
label_path = case_dir / "label.nii.gz"

print("Case:", CASE)
print("Image:", image_path.name)
print("Label:", label_path.name)

## 5 — Load the 3D CBCT and its segmentation

A 2D image is **height × width**.

A CBCT is a **3D volume** made of voxels.

The label is another 3D volume aligned with the CBCT.

In [ ]:
img_nii = nib.load(image_path)
lab_nii = nib.load(label_path)

image = img_nii.get_fdata()
label = lab_nii.get_fdata()

print("CBCT shape:", image.shape)
print("Label shape:", label.shape)
print("Voxel spacing (mm):", img_nii.header.get_zooms()[:3])
print("Label values:", np.unique(label))

assert image.shape == label.shape, "Image and label dimensions do not match."
print("✓ Image and label dimensions match.")

## 6 — View the same CBCT in 3 planes

A CBCT should be inspected in:
- axial
- coronal
- sagittal

This is one reason a 3D volume should not be treated as hundreds of unrelated 2D images.

In [ ]:
x, y, z = [s // 2 for s in image.shape]

fig, ax = plt.subplots(1, 3, figsize=(15, 5))

ax[0].imshow(image[:, :, z].T, cmap="gray", origin="lower")
ax[0].set_title("Axial")

ax[1].imshow(image[:, y, :].T, cmap="gray", origin="lower")
ax[1].set_title("Coronal")

ax[2].imshow(image[x, :, :].T, cmap="gray", origin="lower")
ax[2].set_title("Sagittal")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

## 7 — Overlay the segmentation

The label occupies the same 3D space as the CBCT.

For a simple visual check, we display the middle axial slice and overlay every non-zero label voxel.

In [ ]:
mask = label > 0
z = image.shape[2] // 2

plt.figure(figsize=(7, 7))
plt.imshow(image[:, :, z].T, cmap="gray", origin="lower")
plt.imshow(mask[:, :, z].T, cmap="autumn", alpha=0.35, origin="lower")
plt.title(f"{CASE} — CBCT + segmentation overlay")
plt.axis("off")
plt.show()

## 8 — How would we annotate this in 3D Slicer?

For a new real CBCT:

1. Import the de-identified CBCT into **3D Slicer**.
2. Open **Segment Editor**.
3. Create the structure to annotate.
4. Use Paint / Threshold / Grow from Seeds / Fill Between Slices where appropriate.
5. Check axial, coronal, sagittal, and 3D views.
6. Correct the mask manually.
7. Perform expert QC.
8. Export the image and matching segmentation.

### Critical rule

**Keep image and label spatially aligned.**

## 9 — Patient-level splitting

The workshop copy contains **9 independent case folders**.

For teaching, here is a simple split:

- 6 cases → training
- 1 case → validation
- 2 cases → test

The exact split is only a demonstration.

In [ ]:
train_cases = cases[:6]
val_cases   = cases[6:7]
test_cases  = cases[7:9]

print("TRAIN:", [p.name for p in train_cases])
print("VALIDATION:", [p.name for p in val_cases])
print("TEST:", [p.name for p in test_cases])

assert set(train_cases).isdisjoint(test_cases)
print("\n✓ No case appears in both train and test.")

## Why not split CBCT slices randomly?

### ❌ Wrong
Slices from the same patient are divided between train and test.

### ✅ Correct principle
**One patient / one volume belongs to one split only.**

Otherwise, the model may be tested on anatomy it effectively already saw during training.

## 10 — Where would AI model training fit?

For segmentation:

**3D CBCT → 3D segmentation model → 3D mask**

Examples include:
- 3D U-Net
- nnU-Net
- MONAI-based workflows

For this workshop, we intentionally stop before 3D model training.

The main Session 2 training exercise remains:

**2D implant recognition in Colab → Landing AI**

This short CBCT lab focuses only on understanding **real 3D data, annotation, geometry, and patient-level splitting**.

## ✅ Take-home message

**2D dental image**
→ pixels → image label / box / 2D mask

**3D CBCT**
→ voxels → 3D mask → preserve geometry → split by patient

### Remember

**Understand the 3D data correctly before training a 3D model.**